# AKC-Tutor — Phase III: Knowledge-State Estimation and Evidence-Gated Affect Fusion

**Run all in Google Colab (CPU sufficient for tabular baselines).** This notebook downloads a public educational interaction dataset, constructs *past-only* knowledge and behavioral indicators, trains multiple baselines, performs learner-disjoint testing, calculates bootstrap confidence intervals, produces charts, and exports a results ZIP.

**Scientific boundary:** Behavioral indicators are **not validated emotion labels**. The default run is **knowledge + behavioral-proxy fusion**, *not* true knowledge–affect fusion. True affect fusion requires matched dialogue/emotion observations for the **same learner and interaction**. Unmatched GoEmotions labels are never joined to educational records.

If the public dataset URL changes, the notebook stops with a clear upload instruction rather than fabricating results.

In [ ]:
# 1. Install dependencies without force-reinstalling NumPy (avoids Colab binary conflicts)
%pip -q install pandas numpy scipy scikit-learn matplotlib seaborn requests joblib

In [ ]:
# 2. Imports and reproducibility
import os, json, time, warnings, zipfile, hashlib
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score, log_loss, brier_score_loss, precision_score, recall_score, confusion_matrix, classification_report
from sklearn.utils import shuffle
import requests, joblib
SEED=42; np.random.seed(SEED)
ROOT=Path('/content/akc_phase3'); ROOT.mkdir(exist_ok=True,parents=True)
OUT=ROOT/'results'; OUT.mkdir(exist_ok=True)
print('Ready:',ROOT)

## 3. Download a real educational interaction dataset

The primary source is the public `as.csv` example hosted by the pyBKT project (ASSISTments-derived records). If the file is unavailable, upload an educational interactions CSV with learner ID, skill ID, and correctness. No synthetic fallback is used.

**Note:** The example dataset may differ from a complete official ASSISTments release. The exact downloaded file, row count, and SHA-256 are recorded.

In [ ]:
# 3. Download; optional manual upload if source has moved
URLS=[
 'https://raw.githubusercontent.com/CAHLR/pyBKT-examples/master/data/as.csv',
 'https://raw.githubusercontent.com/CAHLR/pyBKT-examples/main/data/as.csv',
]
DATA=ROOT/'educational_interactions.csv'; downloaded=False; errors=[]
for url in URLS:
 try:
  r=requests.get(url,timeout=45)
  if r.ok and len(r.content)>1000 and b',' in r.content[:1000]:
   DATA.write_bytes(r.content); downloaded=True; source=url; break
  errors.append(f'{url}: HTTP {r.status_code}, bytes={len(r.content)}')
 except Exception as e: errors.append(f'{url}: {type(e).__name__}: {e}')
if not downloaded:
 print('Automatic download unavailable. Attempting interactive CSV upload.')
 print('Source errors:',errors)
 from google.colab import files
 uploaded=files.upload()
 csvs=[(name,content) for name,content in uploaded.items() if name.lower().endswith('.csv')]
 if not csvs: raise RuntimeError('No CSV supplied. Upload real learner-interaction data; synthetic data are prohibited.')
 DATA.write_bytes(csvs[0][1]); source='user_upload:'+csvs[0][0]
sha=hashlib.sha256(DATA.read_bytes()).hexdigest()
df=pd.read_csv(DATA,low_memory=False)
print('Source:',source,'SHA256:',sha,'Rows:',len(df),'Columns:',list(df.columns))
assert len(df)>100, 'Dataset too small for reliable evaluation.' 

In [ ]:
# 4. Resolve source schema; never silently invent columns
print('Available columns:',df.columns.tolist())
def pick(names):
 normalized={str(c).lower().strip().replace(' ','_'):c for c in df.columns}
 for name in names:
  key=name.lower().strip().replace(' ','_')
  if key in normalized: return normalized[key]
 return None
uid=pick(['user_id','user','student_id','student','anon_student_id','studentid'])
skill=pick(['skill_name','skill_id','skill','kc(default)','kc','concept','problem_id'])
correct=pick(['correct','correctness','is_correct','answered_correctly','outcome'])
timecol=pick(['order_id','timestamp','start_time','first_action','row','sequence_id','problem_log_id','problem_id'])
latency=pick(['ms_first_response','response_time','latency','elapsed_time'])
print('Mapped: user=',uid,'skill=',skill,'correct=',correct,'time/order=',timecol,'latency=',latency)
if not all([uid,skill,correct]):
 raise ValueError('Required learner/skill/correct columns not found. Supply a CSV with user_id, skill_id, correct (0/1).')
work=pd.DataFrame({'user_id':df[uid].astype(str),'skill_id':df[skill].astype(str), 'correct':pd.to_numeric(df[correct],errors='coerce')})
work['row_order']=np.arange(len(work))
if timecol is not None:
 parsed=pd.to_datetime(df[timecol],errors='coerce',utc=True)
 if parsed.notna().mean()>.85: work['event_order']=parsed
 else:
  numeric=pd.to_numeric(df[timecol],errors='coerce')
  work['event_order']=numeric if numeric.notna().mean()>.85 else work['row_order']
else: work['event_order']=work['row_order']
if latency is not None: work['response_time']=pd.to_numeric(df[latency],errors='coerce')
work=work[work.correct.isin([0,1]) & work.user_id.ne('nan') & work.skill_id.ne('nan')].copy()
work['correct']=work.correct.astype(int)
work=work.sort_values(['user_id','event_order','row_order'],kind='mergesort').reset_index(drop=True)
print('Cleaned:',len(work),'Learners:',work.user_id.nunique(),'Skills:',work.skill_id.nunique())
if work.user_id.nunique()<20: raise ValueError('Too few distinct learners for learner-disjoint evaluation.')
work.to_csv(OUT/'cleaned_interactions.csv',index=False)

## 5. Leakage-safe features

Each feature is computed using **previous interactions only**. No current correctness, future answer, or test outcome enters the input features. We create (a) knowledge-history features and (b) behavioral engagement **proxies**, not emotion labels. The optional matched-affect pathway is separate.

In [ ]:
# 5. Past-only knowledge and behavioral features
# Grouped cumulative sums exclude current outcome using subtraction; this is legitimate historical information.
g=work.groupby('user_id',sort=False)
work['prior_attempts']=g.cumcount()
work['prior_correct']=g['correct'].cumsum()-work['correct']
work['prior_accuracy']=work['prior_correct']/work['prior_attempts'].replace(0,np.nan)
sg=work.groupby(['user_id','skill_id'],sort=False)
work['prior_skill_attempts']=sg.cumcount()
work['prior_skill_correct']=sg['correct'].cumsum()-work['correct']
work['prior_skill_accuracy']=work['prior_skill_correct']/work['prior_skill_attempts'].replace(0,np.nan)
work['prev_correct']=g['correct'].shift(1)
work['prev2_correct']=g['correct'].shift(2)
work['recent2_accuracy']=(work['prev_correct']+work['prev2_correct'])/2
work['prev_skill_correct']=sg['correct'].shift(1)
# Behavioral proxies: recent error pattern and optional previous response time
work['recent_error_proxy']=1-work['recent2_accuracy']
work['recent_change_proxy']=work['prev_correct']-work['prev2_correct']
behavior=['recent_error_proxy','recent_change_proxy']
if 'response_time' in work:
 work['previous_response_time']=g['response_time'].shift(1)
 work['previous_response_time']=np.log1p(work['previous_response_time'].clip(lower=0))
 behavior.append('previous_response_time')
knowledge=['prior_attempts','prior_correct','prior_accuracy','prior_skill_attempts','prior_skill_correct','prior_skill_accuracy','prev_skill_correct']
print('Knowledge features:',knowledge,'Behavioral proxies:',behavior)
assert not work[knowledge+behavior].empty
# Save metadata: these are behavioral indicators, NOT validated affect states.
(OUT/'feature_definitions.json').write_text(json.dumps({'knowledge_features':knowledge,'behavioral_proxy_features':behavior,'warning':'Behavioral proxies are not emotion labels or measured learner affect.'},indent=2))

## 6. Optional TRUE affect observations (disabled by default)

If you have a **matched** CSV with `user_id`, `row_order`, and an independently observed `affect_score` (numeric), upload it and set `MATCHED_AFFECT_CSV` to its path. `row_order` refers to the original input CSV row number (0-based). Affect must be measured **before the target response**. The notebook will otherwise report **no true-affect experiment**. For an emotion classifier, obtain its predictions from genuine learner dialogue and align them to those same interactions first.

Do not attach labels from GoEmotions or DAiSEE to unrelated ASSISTments learners.

In [ ]:
# 6. Optional matched-affect input; default None for fully automatic real-data baseline
MATCHED_AFFECT_CSV=None  # e.g. '/content/matched_affect.csv'
true_affect=False
if MATCHED_AFFECT_CSV:
 a=pd.read_csv(MATCHED_AFFECT_CSV)
 required={'user_id','row_order','affect_score'}
 if not required.issubset(a.columns): raise ValueError(f'Matched affect CSV must include {required}')
 if a.duplicated(['user_id','row_order']).any(): raise ValueError('Duplicate matched affect keys')
 a['user_id']=a['user_id'].astype(str)
 a['affect_score']=pd.to_numeric(a.affect_score,errors='coerce')
 if a.affect_score.notna().sum()<100: raise ValueError('Insufficient measured affect observations')
 work=work.merge(a[['user_id','row_order','affect_score']],on=['user_id','row_order'],how='left',validate='one_to_one')
 # only matched examples for a fair comparison; do not impute unmatched emotion labels
 work=work[work.affect_score.notna()].copy()
 true_affect=True
 print('True-affect matched interactions:',len(work))
else: print('True affect mode DISABLED: no matched learner-affect observations provided.')

In [ ]:
# 7. Learner-disjoint train/validation/test split, 70/15/15 approx
ids=np.array(sorted(work.user_id.unique()))
rng=np.random.default_rng(SEED); rng.shuffle(ids)
n=len(ids); ntrain=int(.70*n); nval=int(.15*n)
train_ids=set(ids[:ntrain]); val_ids=set(ids[ntrain:ntrain+nval]); test_ids=set(ids[ntrain+nval:])
parts={'train':work[work.user_id.isin(train_ids)].copy(), 'validation':work[work.user_id.isin(val_ids)].copy(), 'test':work[work.user_id.isin(test_ids)].copy()}
assert all(len(p)>0 for p in parts.values())
assert not (train_ids & val_ids or train_ids & test_ids or val_ids & test_ids)
for k,v in parts.items():print(k,'rows',len(v),'learners',v.user_id.nunique(),'positive rate',round(v.correct.mean(),3))
# Historical features in validation/test use only each learner's past within the same partition.
# We report all interactions including cold-start examples; missing history is imputed.
(OUT/'split_summary.json').write_text(json.dumps({k:{'rows':len(v),'learners':int(v.user_id.nunique()),'positive_rate':float(v.correct.mean())} for k,v in parts.items()},indent=2))

## 8. Model training and ablation

All variants use the **same learner-disjoint partitions**. Models are regularized logistic regression with median imputation, missing-value indicators, and standardization. The shuffled-behavior negative control tests whether meaningful alignment of the proxy features matters. This is **not** a test of true emotion awareness.

In [ ]:
# 8. Train and evaluate all variants
variants={'knowledge_only':knowledge,'behavior_proxy_only':behavior,'knowledge_plus_behavior_proxy':knowledge+behavior}
if true_affect:
 variants['knowledge_plus_measured_affect']=knowledge+['affect_score']
 variants['measured_affect_only']=['affect_score']
models={}; results=[]; preds={}
def model_factory(C):
 return Pipeline([('imputer',SimpleImputer(strategy='median',add_indicator=True)),('scaler',StandardScaler()),('clf',LogisticRegression(C=C,max_iter=1200,random_state=SEED))])
def metric(y,p):
 pred=(p>=.5).astype(int)
 return {'auc':float(roc_auc_score(y,p)) if len(np.unique(y))>1 else None,'accuracy':float(accuracy_score(y,pred)),'f1':float(f1_score(y,pred,zero_division=0)),'log_loss':float(log_loss(y,p,labels=[0,1])),'brier':float(brier_score_loss(y,p))}
for name,cols in variants.items():
 best=None
 for C in [.01,.1,1.,10.]:
  m=model_factory(C); m.fit(parts['train'][cols],parts['train'].correct)
  vp=m.predict_proba(parts['validation'][cols])[:,1]
  score=metric(parts['validation'].correct,vp)['auc']
  if best is None or (score is not None and score>best[0]): best=(score,C,m)
 _,C,m=best
 p=m.predict_proba(parts['test'][cols])[:,1]
 scores=metric(parts['test'].correct,p)
 results.append({'model':name,'C':C,**scores});models[name]=m;preds[name]=p
 joblib.dump(m,OUT/f'{name}.joblib')
 print(name,'C=',C,'test AUC=',scores['auc'],'F1=',scores['f1'])
# Shuffled behavioral negative control: permute behavioral proxies separately within each split,
# destroying alignment while retaining their marginal distribution.
rng=np.random.default_rng(SEED+10)
shuffled={}
for split,frame in parts.items():
 t=frame[knowledge+behavior].copy()
 for col in behavior:t[col]=rng.permutation(t[col].to_numpy())
 shuffled[split]=t
best=None
for C in [.01,.1,1.,10.]:
 m=model_factory(C);m.fit(shuffled['train'],parts['train'].correct)
 v=m.predict_proba(shuffled['validation'])[:,1]; score=metric(parts['validation'].correct,v)['auc']
 if best is None or (score is not None and score>best[0]):best=(score,C,m)
_,C,m=best;p=m.predict_proba(shuffled['test'])[:,1]
results.append({'model':'knowledge_plus_shuffled_behavior_control','C':C,**metric(parts['test'].correct,p)})
preds['knowledge_plus_shuffled_behavior_control']=p
joblib.dump(m,OUT/'knowledge_plus_shuffled_behavior_control.joblib')
# Majority-class predictor based ONLY on training prevalence
prevalence=float(parts['train'].correct.mean());p=np.full(len(parts['test']),prevalence)
results.append({'model':'constant_training_prevalence','C':None,**metric(parts['test'].correct,p)})
preds['constant_training_prevalence']=p
res=pd.DataFrame(results).sort_values('auc',ascending=False)
res.to_csv(OUT/'model_comparison.csv',index=False)
display(res)

In [ ]:
# 9. Export test predictions with learner keys (no fabricated affect labels)
test=parts['test'][['user_id','skill_id','row_order','correct']].copy()
for name,p in preds.items():test['p_'+name]=p
test.to_csv(OUT/'test_predictions.csv',index=False)
# 95% learner-cluster bootstrap for AUC differences; samples whole learners (not individual rows).
N_BOOT=500
base='knowledge_only';fusion='knowledge_plus_behavior_proxy'
y=parts['test'].correct.to_numpy();users=parts['test'].user_id.to_numpy()
unique=np.unique(users); indices={u:np.flatnonzero(users==u) for u in unique}
rr=np.random.default_rng(SEED+2); boot=[]
for i in range(N_BOOT):
 sampled=rr.choice(unique,size=len(unique),replace=True)
 ix=np.concatenate([indices[u] for u in sampled]); yy=y[ix]
 if len(np.unique(yy))<2:continue
 boot.append(roc_auc_score(yy,preds[fusion][ix])-roc_auc_score(yy,preds[base][ix]))
ci=np.quantile(boot,[.025,.975]).tolist() if boot else [None,None]
delta=float(roc_auc_score(y,preds[fusion])-roc_auc_score(y,preds[base]))
comparison={'delta_auc_fusion_minus_knowledge':delta,'learner_cluster_bootstrap_95pct_ci':ci,'bootstrap_replicates':len(boot),'interpretation':'Behavioral-proxy fusion only; no measured affect unless matched data supplied.'}
(OUT/'paired_bootstrap.json').write_text(json.dumps(comparison,indent=2))
print('Fusion minus knowledge AUC:',delta,'95% learner-cluster bootstrap CI:',ci)

In [ ]:
# 10. Research-ready figures
plt.figure(figsize=(10,5));plot=res.sort_values('auc',ascending=True)
plt.barh(plot.model,plot.auc);plt.xlabel('Held-out ROC-AUC');plt.title('AKC-Tutor Phase III: Learner-disjoint evaluation');plt.tight_layout();plt.savefig(OUT/'auc_comparison.png',dpi=180);plt.show()
plt.figure(figsize=(10,5));plot=res.sort_values('f1',ascending=True)
plt.barh(plot.model,plot.f1);plt.xlabel('Held-out F1 (threshold = 0.5)');plt.title('Ablation comparison');plt.tight_layout();plt.savefig(OUT/'f1_comparison.png',dpi=180);plt.show()
for name in [base,fusion]:
 pred=(preds[name]>=.5).astype(int)
 cm=confusion_matrix(y,pred,labels=[0,1]);plt.figure(figsize=(4.5,4))
 sns.heatmap(cm,annot=True,fmt='d',cmap='Blues',xticklabels=['Incorrect','Correct'],yticklabels=['Incorrect','Correct'])
 plt.xlabel('Predicted');plt.ylabel('Actual');plt.title(name);plt.tight_layout();plt.savefig(OUT/f'confusion_{name}.png',dpi=180);plt.show()
 pd.DataFrame(classification_report(y,pred,output_dict=True,zero_division=0)).T.to_csv(OUT/f'classification_{name}.csv')

In [ ]:
# 11. Reproducibility and honest research interpretation
import sklearn, scipy
metadata={'dataset_source':source,'dataset_sha256':sha,'rows_raw':int(len(df)),'rows_clean':int(len(work)),
 'num_learners':int(work.user_id.nunique()),'num_skills':int(work.skill_id.nunique()),
 'python_seed':SEED,'scikit_learn_version':sklearn.__version__,'pandas_version':pd.__version__,
 'split':'learner-disjoint approximately 70/15/15','history_features':'past-only per-learner and per-skill',
 'true_affect_mode':true_affect,'behavior_proxy_is_emotion':False,
 'primary_endpoint':'held-out next-response correctness ROC-AUC',
 'warning':'Do not describe behavior-proxy fusion as emotion-aware fusion. Real affect requires aligned independent learner observations.'}
(OUT/'run_metadata.json').write_text(json.dumps(metadata,indent=2))
readme="""AKC-Tutor Phase III experimental outputs

Default mode: real learner interaction data, past-only knowledge features and behavioral proxies.
Behavioral proxies are NOT emotion labels. No GoEmotions samples were paired to learners.
The output measures next-response correctness prediction, NOT conversational tutoring quality.
True affect mode only runs if independently measured, learner-and-interaction-aligned affect data are provided.
Compare model_comparison.csv and paired_bootstrap.json; use run_metadata.json for dataset provenance.
"""
(OUT/'README.txt').write_text(readme)
archive=ROOT/'AKC_Tutor_Phase3_Results.zip'
with zipfile.ZipFile(archive,'w',compression=zipfile.ZIP_DEFLATED) as z:
 for f in OUT.iterdir():
  if f.is_file():z.write(f,arcname=f.name)
print('Saved:',archive,'size MB:',round(archive.stat().st_size/1e6,2))
from google.colab import files
files.download(str(archive))

## What to send back

Upload **`AKC_Tutor_Phase3_Results.zip`**. We will inspect held-out AUC, F1, learner-cluster confidence intervals, and ablation differences before moving to Phase IV. If the optional matched-affect dataset was not provided, we will explicitly label this a **behavioral-proxy pilot** and not claim that genuine learner emotion fusion was evaluated.

**Research caveat:** Historical student outcomes and recent-error indicators overlap conceptually. Improvements, if any, establish predictive utility of added historical features, not independent affect recognition. The next phase requires genuine learner dialogue or a suitably aligned educational affect dataset.